# Short-Horizon Return Prediction - iRage Kaggle Competition
Complete pipeline with EDA, drift detection, multiple models, and ensemble

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)

print("Libraries loaded successfully!")

## 2. Load Data from Parquet Files

In [ ]:
# Load training and test data
train_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/train.parquet')
test_df = pd.read_parquet('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/test.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns sample: {train_df.columns[:10].tolist()}")
print(f"\nTarget column exists: {'TARGET' in train_df.columns}")

## 3. Data Quality Check

In [ ]:
# Check missing values
print("Missing values in train:")
missing_train = train_df.isnull().sum()
print(missing_train[missing_train > 0].head(10))

print("\nMissing values in test:")
missing_test = test_df.isnull().sum()
print(missing_test[missing_test > 0].head(10))

# Fill NaNs with median or 0
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

print("\nNaNs filled with median values.")

## 4. Target Variable Analysis

In [ ]:
# Analyze target
target = train_df['TARGET']

print("Target Statistics:")
print(f"Mean: {target.mean():.6f}")
print(f"Std: {target.std():.6f}")
print(f"Min: {target.min():.6f}")
print(f"Max: {target.max():.6f}")
print(f"Median: {target.median():.6f}")
print(f"Skewness: {target.skew():.4f}")
print(f"Kurtosis: {target.kurtosis():.4f}")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(target, bins=50, edgecolor='black')
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('TARGET')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(target)
axes[1].set_title('Target Boxplot')
axes[1].set_ylabel('TARGET')
plt.tight_layout()
plt.show()

# Check outliers
q1 = target.quantile(0.25)
q3 = target.quantile(0.75)
iqr = q3 - q1
outliers = ((target < q1 - 1.5*iqr) | (target > q3 + 1.5*iqr)).sum()
print(f"\nOutliers (IQR method): {outliers} ({100*outliers/len(target):.2f}%)")

## 5. Feature Engineering & Selection

In [ ]:
# Separate ID and TARGET from features
X_train = train_df.drop(['ID', 'TARGET'], axis=1)
y_train = train_df['TARGET']
X_test = test_df.drop(['ID'], axis=1)
test_ids = test_df['ID'].copy()

print(f"Features in X_train: {X_train.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Check for constant columns
const_cols = X_train.columns[X_train.nunique() == 1].tolist()
if const_cols:
    print(f"\nRemoving {len(const_cols)} constant columns")
    X_train = X_train.drop(const_cols, axis=1)
    X_test = X_test.drop(const_cols, axis=1)

print(f"\nFeatures after cleaning: {X_train.shape[1]}")

## 6. Train-Test Shift Detection (Adversarial Validation)

In [ ]:
# Create binary classification task: 0=train, 1=test
X_shift = pd.concat([X_train, X_test], axis=0, ignore_index=True)
y_shift = np.concatenate([np.zeros(len(X_train)), np.ones(len(X_test))])

# Train adversarial classifier
rf_shift = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
shift_auc = cross_val_score(rf_shift, X_shift, y_shift, cv=5, scoring='roc_auc').mean()

print(f"Adversarial Validation AUC: {shift_auc:.4f}")
print(f"Interpretation:")
print(f"  0.50 = No shift (train=test)")
print(f"  0.60+ = Moderate shift")
print(f"  0.70+ = Strong shift")

if shift_auc > 0.65:
    print(f"\n⚠️ WARNING: SIGNIFICANT DISTRIBUTION SHIFT DETECTED!")
    print(f"Private leaderboard may differ greatly from public.")
else:
    print(f"\n✓ Shift is moderate; CV strategy should work reasonably.")

## 7. Validation Strategy Setup

In [ ]:
# Use Repeated KFold for robust validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store out-of-fold predictions for ensembling
oof_predictions = {}

print(f"Validation Strategy: 5-Fold Cross-Validation")
print(f"Total training samples: {len(X_train)}")
print(f"Samples per fold (approx): {len(X_train) // 5}")

## 8. Model 1: LightGBM

In [ ]:
# LightGBM Model
oof_lgb = np.zeros(len(X_train))
test_pred_lgb = np.zeros(len(X_test))
lgb_scores = []

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_data_in_leaf': 20,
    'lambda_l1': 0.5,
    'lambda_l2': 0.5,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'random_state': 42,
    'verbose': -1
}

fold_num = 0
for train_idx, val_idx in kf.split(X_train):
    fold_num += 1
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Train
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val)
    
    model_lgb = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    # OOF
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    
    # Test
    test_pred_lgb += model_lgb.predict(X_test) / 5
    
    # Score
    r2_val = r2_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(r2_val)
    print(f"Fold {fold_num}/{5} | LightGBM R² = {r2_val:.6f}")

print(f"\nLightGBM CV R²: {np.mean(lgb_scores):.6f} (+/- {np.std(lgb_scores):.6f})")
oof_predictions['lgb'] = oof_lgb

## 9. Model 2: CatBoost

In [ ]:
# CatBoost Model
oof_cb = np.zeros(len(X_train))
test_pred_cb = np.zeros(len(X_test))
cb_scores = []

cb_params = {
    'iterations': 500,
    'depth': 6,
    'learning_rate': 0.05,
    'l2_leaf_reg': 3,
    'random_state': 42,
    'verbose': False,
    'task_type': 'CPU'
}

fold_num = 0
for train_idx, val_idx in kf.split(X_train):
    fold_num += 1
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Train
    model_cb = cb.CatBoostRegressor(**cb_params)
    model_cb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )
    
    # OOF
    oof_cb[val_idx] = model_cb.predict(X_val)
    
    # Test
    test_pred_cb += model_cb.predict(X_test) / 5
    
    # Score
    r2_val = r2_score(y_val, oof_cb[val_idx])
    cb_scores.append(r2_val)
    print(f"Fold {fold_num}/{5} | CatBoost R² = {r2_val:.6f}")

print(f"\nCatBoost CV R²: {np.mean(cb_scores):.6f} (+/- {np.std(cb_scores):.6f})")
oof_predictions['cb'] = oof_cb

## 10. Model 3: Ridge Regression (Linear Baseline)

In [ ]:
# Ridge Regression
oof_ridge = np.zeros(len(X_train))
test_pred_ridge = np.zeros(len(X_test))
ridge_scores = []

# Scale features for linear model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

fold_num = 0
for train_idx, val_idx in kf.split(X_train_scaled):
    fold_num += 1
    
    X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Train
    model_ridge = Ridge(alpha=1.0, random_state=42)
    model_ridge.fit(X_tr, y_tr)
    
    # OOF
    oof_ridge[val_idx] = model_ridge.predict(X_val)
    
    # Test
    test_pred_ridge += model_ridge.predict(X_test_scaled) / 5
    
    # Score
    r2_val = r2_score(y_val, oof_ridge[val_idx])
    ridge_scores.append(r2_val)
    print(f"Fold {fold_num}/{5} | Ridge R² = {r2_val:.6f}")

print(f"\nRidge CV R²: {np.mean(ridge_scores):.6f} (+/- {np.std(ridge_scores):.6f})")
oof_predictions['ridge'] = oof_ridge

## 11. Model Comparison

In [ ]:
# Compare models
model_comparison = pd.DataFrame({
    'Model': ['LightGBM', 'CatBoost', 'Ridge'],
    'Mean R²': [np.mean(lgb_scores), np.mean(cb_scores), np.mean(ridge_scores)],
    'Std R²': [np.std(lgb_scores), np.std(cb_scores), np.std(ridge_scores)],
    'Min R²': [np.min(lgb_scores), np.min(cb_scores), np.min(ridge_scores)],
    'Max R²': [np.max(lgb_scores), np.max(cb_scores), np.max(ridge_scores)]
})

print("\n" + "="*70)
print("MODEL COMPARISON (Cross-Validation Results)")
print("="*70)
print(model_comparison.to_string(index=False))
print("="*70)

## 12. Ensemble Strategy

In [ ]:
# Weighted ensemble based on CV performance
lgb_weight = np.mean(lgb_scores) / (np.mean(lgb_scores) + np.mean(cb_scores) + np.mean(ridge_scores))
cb_weight = np.mean(cb_scores) / (np.mean(lgb_scores) + np.mean(cb_scores) + np.mean(ridge_scores))
ridge_weight = np.mean(ridge_scores) / (np.mean(lgb_scores) + np.mean(cb_scores) + np.mean(ridge_scores))

print(f"Ensemble Weights:")
print(f"  LightGBM: {lgb_weight:.4f}")
print(f"  CatBoost: {cb_weight:.4f}")
print(f"  Ridge:    {ridge_weight:.4f}")

# OOF ensemble
oof_ensemble = lgb_weight * oof_lgb + cb_weight * oof_cb + ridge_weight * oof_ridge
ensemble_r2 = r2_score(y_train, oof_ensemble)

print(f"\nEnsemble OOF R²: {ensemble_r2:.6f}")

# Test predictions
test_pred_ensemble = lgb_weight * test_pred_lgb + cb_weight * test_pred_cb + ridge_weight * test_pred_ridge

print(f"\nTest Predictions Stats:")
print(f"  Mean: {test_pred_ensemble.mean():.6f}")
print(f"  Std:  {test_pred_ensemble.std():.6f}")
print(f"  Min:  {test_pred_ensemble.min():.6f}")
print(f"  Max:  {test_pred_ensemble.max():.6f}")

## 13. Generate Submission

In [ ]:
# Create submission file
submission = pd.DataFrame({
    'ID': test_ids.values,
    'TARGET': test_pred_ensemble
})

print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

# Save to submission.csv
submission.to_csv('submission.csv', index=False)

print(f"\n✓ Submission saved as 'submission.csv'")
print(f"\nSubmission file ready for upload!")

## 14. Diagnostics & Summary

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"\nTraining Samples: {len(X_train):,}")
print(f"Test Samples: {len(X_test):,}")
print(f"Features Used: {X_train.shape[1]}")
print(f"\nDistribution Shift (Adversarial AUC): {shift_auc:.4f}")
print(f"\nBest Single Model: LightGBM (R² = {np.mean(lgb_scores):.6f})")
print(f"Ensemble R² (OOF): {ensemble_r2:.6f}")
print(f"\nModel Weights in Ensemble:")
print(f"  - LightGBM: {lgb_weight*100:.1f}%")
print(f"  - CatBoost: {cb_weight*100:.1f}%")
print(f"  - Ridge:    {ridge_weight*100:.1f}%")
print(f"\nExpected Test R² (from CV): ~{ensemble_r2:.4f}")
print(f"(may be lower on private test due to {shift_auc:.2f} shift)")
print("\n" + "="*70)